In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
colombia_df = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Drug Seizures (UNODC)/Raw/Aggregated colombia.csv")

<ipython-input-129-8f0f9cde793b>:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  colombia_df = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Drug Seizures (UNODC)/Raw/Aggregated colombia.csv")


###2012

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2012]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

In [ ]:
remove = ['Panama', 'Costa Rica', 'Belgica', 'Estados Unidos', 'Paises Bajos', 'Nicaragua', 'Veraguas', 'España']
df_2012 = df_2012[~df_2012['Administrative Region'].isin(remove)]
df_2012["Administrative Region"] = df_2012["Administrative Region"].replace("Providencia Y Santa Catalina", "Archipielago De San Andres, Providencia Y Santa Catalina")

departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)

<ipython-input-59-ff8bcd087a96>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].replace("Providencia Y Santa Catalina", "Archipielago De San Andres, Providencia Y Santa Catalina")
<ipython-input-59-ff8bcd087a96>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-59-ff8bcd087a96>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

In [ ]:
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

In [ ]:
df_2012['Department'] = None

In [ ]:
departments['DEPARTAMENTO'].unique()

array(['Antioquia', 'Boyacá', 'Córdoba', 'Chocó', 'Nariño', 'Santander',
       'Meta', 'Atlántico', 'Bolívar', 'Caldas', 'Caquetá', 'Cauca',
       'Cesar', 'Cundinamarca', 'Huila', 'La Guajira', 'Magdalena',
       'Quindío', 'Risaralda', 'Sucre', 'Tolima', 'Arauca', 'Casanare',
       'Putumayo', 'Amazonas', 'Guainía', 'Vaupés', 'Vichada', 'Guaviare',
       'Archipiélago de San Andrés, Providencia y Santa Catalina',
       'Bogotá D.C.', 'Norte de Santander', 'Valle del Cauca'],
      dtype=object)

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
3902,2012-12-31,COL,Non-specified cocaine-type,0.152,kg,cali,valle del cauca,Valle del Cauca
3903,2012-12-31,COL,Non-specified cocaine-type,0.880,kg,orito,putumayo,Putumayo
3904,2012-12-31,COL,Cocaine base,0.345,kg,taraza,antioquia,Antioquia
3905,2012-12-31,COL,Cocaine base,0.100,kg,cartagena de indias,bolivar,Bolívar
3906,2012-12-31,COL,Cocaine base,0.200,kg,palmira,valle del cauca,Valle del Cauca


In [ ]:
df_2012.loc[df_2012['Administrative Region'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'

none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))

781


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
3902,2012-12-31,COL,Non-specified cocaine-type,0.152,kg,cali,valle del cauca,Valle del Cauca
3903,2012-12-31,COL,Non-specified cocaine-type,0.880,kg,orito,putumayo,Putumayo
3904,2012-12-31,COL,Cocaine base,0.345,kg,taraza,antioquia,Antioquia
3905,2012-12-31,COL,Cocaine base,0.100,kg,cartagena de indias,bolivar,Bolívar
3906,2012-12-31,COL,Cocaine base,0.200,kg,palmira,valle del cauca,Valle del Cauca


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

91


array(['archipielago de san andres', 'bogota, d.c.',
       'cartagena de indias', 'costa rica', 'valle del guamuez',
       'nicaragua', 'puerto leguizamo'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'
df_2012.loc[df_2012['City'] == 'archipielago de san andres', 'Department'] = 'Archipiélago de San Andrés, Providencia y Santa Catalina'
df_2012.loc[df_2012['City'] == 'cartagena de indias', 'Department'] = 'Bolívar'
df_2012.loc[df_2012['City'] == 'valle del guamuez', 'Department'] = 'Putumayo'
df_2012.loc[df_2012['City'] == 'puerto leguizamo', 'Department'] = 'Putumayo'
df_2012 = df_2012[df_2012['City'] != 'nicaragua']
df_2012 = df_2012[df_2012['City'] != 'costa rica']

In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows.head()

0


,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department


In [ ]:
df_2012.head()
df_2012.to_csv("/content/drive/MyDrive/AAA/2012.csv", index=False)

###2013

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2013]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


In [ ]:
df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region
8418,2013-12-31,COL,Cocaine base,0.100,kg,Itagui,NaN
8419,2013-12-30,COL,Non-specified cocaine-type,0.310,kg,Anori,NaN
8420,2013-12-30,COL,Non-specified cocaine-type,0.830,kg,Palmira,NaN
8421,2013-12-30,COL,Non-specified cocaine-type,0.530,kg,Popayan,NaN
8422,2013-12-30,COL,Cocaine base,0.200,kg,Medellin,NaN


In [ ]:
departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

<ipython-input-80-9ef3b57b3c7c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-80-9ef3b57b3c7c>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-80-9ef3b57b3c7c>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
8418,2013-12-31,COL,Cocaine base,0.100,kg,itagui,nan,None
8419,2013-12-30,COL,Non-specified cocaine-type,0.310,kg,anori,nan,None
8420,2013-12-30,COL,Non-specified cocaine-type,0.830,kg,palmira,nan,None
8421,2013-12-30,COL,Non-specified cocaine-type,0.530,kg,popayan,nan,None
8422,2013-12-30,COL,Cocaine base,0.200,kg,medellin,nan,None


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
8418,2013-12-31,COL,Cocaine base,0.100,kg,itagui,nan,Antioquia
8419,2013-12-30,COL,Non-specified cocaine-type,0.310,kg,anori,nan,Antioquia
8420,2013-12-30,COL,Non-specified cocaine-type,0.830,kg,palmira,nan,Valle del Cauca
8421,2013-12-30,COL,Non-specified cocaine-type,0.530,kg,popayan,nan,Cauca
8422,2013-12-30,COL,Cocaine base,0.200,kg,medellin,nan,Antioquia


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

490


array(['bogota, d.c.', 'panama', 'dibulla', 'guatemala', ' mar pacifico ',
       'valle del guamuez', 'paris', 'aguas internacionales  ',
       'costa rica', 'puebloviejo', 'miami', 'honduras', 'belgica ',
       'vistahermosa'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'
df_2012.loc[df_2012['City'] == 'dibulla', 'Department'] = 'La Guajira'
df_2012.loc[df_2012['City'] == 'valle del guamuez', 'Department'] = 'Putumayo'
df_2012.loc[df_2012['City'] == 'puebloviejo', 'Department'] = 'Magdalena'
df_2012.loc[df_2012['City'] == 'vistahermosa', 'Department'] = 'Meta'


df_2012 = df_2012[df_2012['City'] != 'panama']
df_2012 = df_2012[df_2012['City'] != 'guatemala']
df_2012 = df_2012[df_2012['City'] != ' mar pacifico ']
df_2012 = df_2012[df_2012['City'] != 'paris']
df_2012 = df_2012[df_2012['City'] != 'aguas internacionales  ']
df_2012 = df_2012[df_2012['City'] != 'costa rica']
df_2012 = df_2012[df_2012['City'] != 'miami']
df_2012 = df_2012[df_2012['City'] != 'honduras']
df_2012 = df_2012[df_2012['City'] != 'belgica ']

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2013.csv", index=False)

###2014

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2014]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


In [ ]:
df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region
13195,2014-12-31,COL,Non-specified cocaine-type,2.929,kg,Rionegro,Antioquia
13196,2014-12-31,COL,Non-specified cocaine-type,0.030,kg,Bucaramanga,Santander
13197,2014-12-31,COL,Non-specified cocaine-type,0.017,kg,Concordia,Antioquia
13198,2014-12-31,COL,Non-specified cocaine-type,0.002,kg,Cúcuta,Norte De Santander
13199,2014-12-31,COL,Non-specified cocaine-type,0.002,kg,Neiva,Huila


In [ ]:
departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
13195,2014-12-31,COL,Non-specified cocaine-type,2.929,kg,rionegro,antioquia,Antioquia
13196,2014-12-31,COL,Non-specified cocaine-type,0.030,kg,bucaramanga,santander,Santander
13197,2014-12-31,COL,Non-specified cocaine-type,0.017,kg,concordia,antioquia,Antioquia
13198,2014-12-31,COL,Non-specified cocaine-type,0.002,kg,cucuta,norte de santander,Norte de Santander
13199,2014-12-31,COL,Non-specified cocaine-type,0.002,kg,neiva,huila,Huila


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
13195,2014-12-31,COL,Non-specified cocaine-type,2.929,kg,rionegro,antioquia,Antioquia
13196,2014-12-31,COL,Non-specified cocaine-type,0.030,kg,bucaramanga,santander,Santander
13197,2014-12-31,COL,Non-specified cocaine-type,0.017,kg,concordia,antioquia,Antioquia
13198,2014-12-31,COL,Non-specified cocaine-type,0.002,kg,cucuta,norte de santander,Norte de Santander
13199,2014-12-31,COL,Non-specified cocaine-type,0.002,kg,neiva,huila,Huila


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

2678


array(['bogota, d.c.', ' mar pacifico ', ' frontera terrestre venezuela ',
       ' aguas juridiccionales ', ' belgica ', ' panama ', 'venezuela ',
       'panama ', ' aguas internacionales ', 'peru '], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'
df_2012 = df_2012[df_2012['City'] != ' mar pacifico ']
df_2012 = df_2012[df_2012['City'] != ' frontera terrestre venezuela ']
df_2012 = df_2012[df_2012['City'] != ' aguas juridiccionales ']
df_2012 = df_2012[df_2012['City'] != ' belgica ']
df_2012 = df_2012[df_2012['City'] != ' panama ']
df_2012 = df_2012[df_2012['City'] != 'panama ']
df_2012 = df_2012[df_2012['City'] != 'venezuela ']
df_2012 = df_2012[df_2012['City'] != ' aguas internacionales ']
df_2012 = df_2012[df_2012['City'] != 'peru ']



In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2014.csv", index=False)

###2015

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2015]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


In [ ]:
df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region
64942,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,San Rafael,Antioquia
64943,2015-12-31,COL,Cocaine base,0.072,kg,Pasto,Nariño
64944,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,Caracolí,Antioquia
64945,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.009,kg,Sincelejo,Sucre
64946,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.005,kg,Concordia,Antioquia


In [ ]:
departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

<ipython-input-102-9ef3b57b3c7c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-102-9ef3b57b3c7c>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-102-9ef3b57b3c7c>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
64942,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,san rafael,antioquia,Antioquia
64943,2015-12-31,COL,Cocaine base,0.072,kg,pasto,narino,Nariño
64944,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,caracoli,antioquia,Antioquia
64945,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.009,kg,sincelejo,sucre,Sucre
64946,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.005,kg,concordia,antioquia,Antioquia


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
64942,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,san rafael,antioquia,Antioquia
64943,2015-12-31,COL,Cocaine base,0.072,kg,pasto,narino,Nariño
64944,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,caracoli,antioquia,Antioquia
64945,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.009,kg,sincelejo,sucre,Sucre
64946,2015-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.005,kg,concordia,antioquia,Antioquia


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

1937


array(['bogota, d.c.', 'nan'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['Administrative Region'].unique()

82


array(['nan'], dtype=object)

In [ ]:
df_2012 = df_2012[df_2012['Department'].notna()]

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2015.csv", index=False)

###2016

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2016]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


In [ ]:
df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region
118996,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,Palmira,Valle Del Cauca
118997,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,Santa Marta,Magdalena
118998,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.100,kg,"Bogotá, D.C.","Bogotá, D.C."
118999,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,Uramita,Antioquia
119000,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,Palmira,Valle Del Cauca


In [ ]:
departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

<ipython-input-117-9ef3b57b3c7c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-117-9ef3b57b3c7c>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-117-9ef3b57b3c7c>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
118996,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,palmira,valle del cauca,Valle del Cauca
118997,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,santa marta,magdalena,Magdalena
118998,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.100,kg,"bogota, d.c.","bogota, d.c.",None
118999,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,uramita,antioquia,Antioquia
119000,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,palmira,valle del cauca,Valle del Cauca


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
118996,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,palmira,valle del cauca,Valle del Cauca
118997,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,santa marta,magdalena,Magdalena
118998,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.100,kg,"bogota, d.c.","bogota, d.c.",None
118999,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,uramita,antioquia,Antioquia
119000,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,palmira,valle del cauca,Valle del Cauca


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

830


array(['bogota, d.c.', 'nan'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['Administrative Region'].unique()

89


array(['nan'], dtype=object)

In [ ]:
df_2012 = df_2012[df_2012['Department'].notna()]

In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
df_2012.head()

0


,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
118996,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,palmira,valle del cauca,Valle del Cauca
118997,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.007,kg,santa marta,magdalena,Magdalena
118998,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.100,kg,"bogota, d.c.","bogota, d.c.",Bogotá D.C.
118999,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.010,kg,uramita,antioquia,Antioquia
119000,2016-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.003,kg,palmira,valle del cauca,Valle del Cauca


In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2016.csv", index=False)

###2017

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2017]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


In [ ]:
departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

<ipython-input-131-9ef3b57b3c7c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-131-9ef3b57b3c7c>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-131-9ef3b57b3c7c>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
141856,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.160,kg,antioquia,retiro,None
141857,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.015,kg,antioquia,la ceja,None
141858,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.013,kg,valle del cauca,buenaventura,None
141859,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.012,kg,norte de santander,cucuta,None
141860,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.001,kg,antioquia,concordia,None


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
141856,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.160,kg,antioquia,retiro,None
141857,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.015,kg,antioquia,la ceja,None
141858,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.013,kg,valle del cauca,buenaventura,None
141859,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.012,kg,norte de santander,cucuta,None
141860,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.001,kg,antioquia,concordia,None


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Department'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
141856,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.160,kg,antioquia,retiro,Antioquia
141857,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.015,kg,antioquia,la ceja,Antioquia
141858,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.013,kg,valle del cauca,buenaventura,Valle del Cauca
141859,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.012,kg,norte de santander,cucuta,Norte de Santander
141860,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.001,kg,antioquia,concordia,Antioquia


In [ ]:
for index, row in df_2012.iterrows():
  city = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
141856,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.160,kg,antioquia,retiro,Antioquia
141857,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.015,kg,antioquia,la ceja,Antioquia
141858,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.013,kg,valle del cauca,buenaventura,Valle del Cauca
141859,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.012,kg,norte de santander,cucuta,Norte de Santander
141860,2017-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",0.001,kg,antioquia,concordia,Antioquia


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['Administrative Region'].unique()

142


array(['costa rica', 'panama', 'republica dominicana', 'ecuador',
       'almeria', 'brasil', 'mexico', 'belgica ', ' mar pacifico ',
       'venezuela', 'el salvador ', 'guatemala', 'rotterdam ',
       'nicaragua', 'antillas holandesas', ' esmeralda', 'francia',
       'honduras'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'

In [ ]:
df_2012 = df_2012[df_2012['Administrative Region'] != 'costa rica']
df_2012 = df_2012[df_2012['Administrative Region'] != 'panama']
df_2012 = df_2012[df_2012['Administrative Region'] != 'republica dominicana']
df_2012 = df_2012[df_2012['Administrative Region'] != 'ecuador']
df_2012 = df_2012[df_2012['Administrative Region'] != 'brasil']
df_2012 = df_2012[df_2012['Administrative Region'] != 'mexico']
df_2012 = df_2012[df_2012['Administrative Region'] != 'belgica ']
df_2012 = df_2012[df_2012['Administrative Region'] != ' mar pacifico ']

df_2012 = df_2012[df_2012['Administrative Region'] != 'venezuela']
df_2012 = df_2012[df_2012['Administrative Region'] != 'el salvador ']
df_2012 = df_2012[df_2012['Administrative Region'] != 'guatemala']
df_2012 = df_2012[df_2012['Administrative Region'] != 'rotterdam ']
df_2012 = df_2012[df_2012['Administrative Region'] != 'nicaragua']
df_2012 = df_2012[df_2012['Administrative Region'] != 'antillas holandesas']
df_2012 = df_2012[df_2012['Administrative Region'] != 'francia']
df_2012 = df_2012[df_2012['Administrative Region'] != 'honduras']

df_2012.loc[df_2012['Administrative Region'] == 'almeria', 'Department'] = 'Atlántico'
df_2012.loc[df_2012['Administrative Region'] == ' esmeralda', 'Department'] = 'Putumayo'

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2017.csv", index=False)

###2018

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2018]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


In [ ]:
departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

<ipython-input-147-9ef3b57b3c7c>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-147-9ef3b57b3c7c>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-147-9ef3b57b3c7c>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
158100,2018-12-31,COL,Cocaine base,7,g,armenia,quindio,Quindío
158101,2018-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",3,g,armenia,quindio,Quindío
158102,2018-12-31,COL,Cocaine base,253,g,bogota d.c.,"bogota, d.c.",None
158103,2018-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",10,g,bucaramanga,santander,Santander
158104,2018-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",5,g,fredonia,antioquia,Antioquia


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
158100,2018-12-31,COL,Cocaine base,7,g,armenia,quindio,Antioquia
158101,2018-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",3,g,armenia,quindio,Antioquia
158102,2018-12-31,COL,Cocaine base,253,g,bogota d.c.,"bogota, d.c.",Bogotá D.C.
158103,2018-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",10,g,bucaramanga,santander,Santander
158104,2018-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",5,g,fredonia,antioquia,Antioquia


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['Administrative Region'].unique()

4


array(['archipielago de san andres'], dtype=object)

In [ ]:
df_2012.loc[df_2012['Administrative Region'] == 'archipielago de san andres', 'Department'] = 'Archipiélago de San Andrés, Providencia y Santa Catalina'


In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2018.csv", index=False)

###2019

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2019]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())


departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


<ipython-input-156-c70ea339e173>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-156-c70ea339e173>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-156-c70ea339e173>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentati

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
223990,2019-12-31,COL,Cocaine base,0.014,kg,barranquilla,nan,None
223991,2019-12-31,COL,Cocaine base,0.018,kg,barranquilla,nan,None
223992,2019-12-31,COL,Cocaine base,0.02,kg,barranquilla,nan,None
223993,2019-12-31,COL,Cocaine base,0.038,kg,barranquilla,nan,None
223994,2019-12-31,COL,Cocaine base,0.0055,kg,caldas,nan,None


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
223990,2019-12-31,COL,Cocaine base,0.014,kg,barranquilla,nan,Atlántico
223991,2019-12-31,COL,Cocaine base,0.018,kg,barranquilla,nan,Atlántico
223992,2019-12-31,COL,Cocaine base,0.02,kg,barranquilla,nan,Atlántico
223993,2019-12-31,COL,Cocaine base,0.038,kg,barranquilla,nan,Atlántico
223994,2019-12-31,COL,Cocaine base,0.0055,kg,caldas,nan,Antioquia


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

563


array(['cartagena de indias', 'tumaco', 'since', 'buga',
       'valle del guamuez', 'cerro de san antonio', 'pacific ocean',
       'ubate', 'carmen de viboral', 'puerto leguizamo', 'vistahermosa',
       'villa de leiva', 'santa fe de antioquia', 'tolu', 'dibulla',
       'manaure balcon del cesar', 'puebloviejo', 'montanita',
       'chachagsi', 'toluviejo'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'
df_2012.loc[df_2012['City'] == 'dibulla', 'Department'] = 'La Guajira'
df_2012.loc[df_2012['City'] == 'valle del guamuez', 'Department'] = 'Putumayo'
df_2012.loc[df_2012['City'] == 'puebloviejo', 'Department'] = 'Magdalena'
df_2012.loc[df_2012['City'] == 'vistahermosa', 'Department'] = 'Meta'



df_2012.loc[df_2012['City'] == 'cartagena de indias', 'Department'] = 'Bolívar'
df_2012.loc[df_2012['City'] == 'tumaco', 'Department'] = 'Nariño'
df_2012.loc[df_2012['City'] == 'since', 'Department'] = 'Sucre'
df_2012.loc[df_2012['City'] == 'buga', 'Department'] = 'Valle del Cauca'
df_2012.loc[df_2012['City'] == 'valle del guamuez', 'Department'] = 'Putumayo'
df_2012.loc[df_2012['City'] == 'cerro de san antonio', 'Department'] = 'Magdalena'
df_2012.loc[df_2012['City'] == 'ubate', 'Department'] = 'Cundinamarca'
df_2012.loc[df_2012['City'] == 'carmen de viboral', 'Department'] = 'Antioquia'
df_2012.loc[df_2012['City'] == 'puerto leguizamo', 'Department'] = 'Putumayo'
df_2012.loc[df_2012['City'] == 'vistahermosa', 'Department'] = 'Meta'

df_2012.loc[df_2012['City'] == 'villa de leiva', 'Department'] = 'Boyacá'
df_2012.loc[df_2012['City'] == 'santa fe de antioquia', 'Department'] = 'Antioquia'
df_2012.loc[df_2012['City'] == 'tolu', 'Department'] = 'Sucre'
df_2012.loc[df_2012['City'] == 'dibulla', 'Department'] = 'La Guajira'
df_2012.loc[df_2012['City'] == 'manaure balcon del cesar', 'Department'] = 'La Guajira'
df_2012.loc[df_2012['City'] == 'puebloviejo', 'Department'] = 'Magdalena'
df_2012.loc[df_2012['City'] == 'montanita', 'Department'] = 'Magdalena'
df_2012.loc[df_2012['City'] == 'chachagsi', 'Department'] = 'Nariño'
df_2012.loc[df_2012['City'] == 'toluviejo', 'Department'] = 'Sucre'


df_2012 = df_2012[df_2012['City'] != 'pacific ocean']

In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

0


array([], dtype=object)

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2019.csv", index=False)

###2020

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2020]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())


departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


<ipython-input-168-1de0b653d0d5>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-168-1de0b653d0d5>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-168-1de0b653d0d5>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentati

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
264516,2020-12-29,COL,Cocaine base,29.7,kg,la montanita,caqueta,Caquetá
264517,2020-12-27,COL,Cocaine base,6.492,kg,villagarzon,putumayo,Putumayo
264518,2020-12-26,COL,Cocaine base,29.755,kg,miraflores,guaviare,Guaviare
264519,2020-12-26,COL,"Cocaine hydrochloride (HCl, powder cocaine)",2.0,kg,tibu,norte de santander,Norte de Santander
264520,2020-12-25,COL,"Cocaine hydrochloride (HCl, powder cocaine)",500.0,kg,el charco,narino,Nariño


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
264516,2020-12-29,COL,Cocaine base,29.7,kg,la montanita,caqueta,Caquetá
264517,2020-12-27,COL,Cocaine base,6.492,kg,villagarzon,putumayo,Putumayo
264518,2020-12-26,COL,Cocaine base,29.755,kg,miraflores,guaviare,Boyacá
264519,2020-12-26,COL,"Cocaine hydrochloride (HCl, powder cocaine)",2.0,kg,tibu,norte de santander,Norte de Santander
264520,2020-12-25,COL,"Cocaine hydrochloride (HCl, powder cocaine)",500.0,kg,el charco,narino,Nariño


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

41


array(['riohacha (ct)', 'cali (ct)', 'san andres (ct)', 'dibulla'],
      dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'riohacha (ct)', 'Department'] = 'La Guajira'
df_2012.loc[df_2012['City'] == 'cali (ct)', 'Department'] = 'Valle del Cauca'
df_2012.loc[df_2012['City'] == 'san andres (ct)', 'Department'] = 'Archipiélago de San Andrés, Providencia y Santa Catalina'
df_2012.loc[df_2012['City'] == 'dibulla', 'Department'] = 'La Guajira'



In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['Administrative Region'].unique()

0


array([], dtype=object)

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2020.csv", index=False)

###2021

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2021]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())


departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


<ipython-input-177-8018e57dad83>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-177-8018e57dad83>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-177-8018e57dad83>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentati

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
267353,2021-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",3.931,kg,"bogota, d.c.","bogota, d.c.",None
267354,2021-12-29,COL,Cocaine base,119.002,kg,choco,riosucio,None
267355,2021-12-29,COL,Cocaine base,48.69,kg,choco,riosucio,None
267356,2021-12-29,COL,Cocaine base,67.625,kg,choco,riosucio,None
267357,2021-12-29,COL,Cocaine base,21.64,kg,guaviare,san jose del guaviare,None


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
267353,2021-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",3.931,kg,"bogota, d.c.","bogota, d.c.",None
267354,2021-12-29,COL,Cocaine base,119.002,kg,choco,riosucio,None
267355,2021-12-29,COL,Cocaine base,48.69,kg,choco,riosucio,None
267356,2021-12-29,COL,Cocaine base,67.625,kg,choco,riosucio,None
267357,2021-12-29,COL,Cocaine base,21.64,kg,guaviare,san jose del guaviare,None


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Department'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
267353,2021-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",3.931,kg,"bogota, d.c.","bogota, d.c.",None
267354,2021-12-29,COL,Cocaine base,119.002,kg,choco,riosucio,Chocó
267355,2021-12-29,COL,Cocaine base,48.69,kg,choco,riosucio,Chocó
267356,2021-12-29,COL,Cocaine base,67.625,kg,choco,riosucio,Chocó
267357,2021-12-29,COL,Cocaine base,21.64,kg,guaviare,san jose del guaviare,Guaviare


In [ ]:
for index, row in df_2012.iterrows():
  city = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
267353,2021-12-31,COL,"Cocaine hydrochloride (HCl, powder cocaine)",3.931,kg,"bogota, d.c.","bogota, d.c.",None
267354,2021-12-29,COL,Cocaine base,119.002,kg,choco,riosucio,Caldas
267355,2021-12-29,COL,Cocaine base,48.69,kg,choco,riosucio,Caldas
267356,2021-12-29,COL,Cocaine base,67.625,kg,choco,riosucio,Caldas
267357,2021-12-29,COL,Cocaine base,21.64,kg,guaviare,san jose del guaviare,Guaviare


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

152


array(['bogota, d.c.'], dtype=object)

In [ ]:
df_2012.loc[df_2012['City'] == 'bogota, d.c.', 'Department'] = 'Bogotá D.C.'

In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

0


array([], dtype=object)

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2021.csv", index=False)

###2022

In [ ]:
colombia_df["Seizure Date"] = pd.to_datetime(colombia_df["Seizure Date"], format="%d/%m/%Y")
df_2012 = colombia_df[colombia_df["Seizure Date"].dt.year == 2022]

departments = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Data Collection Exploration/Departments and Municipalities (gov.co datos abiertos)/Departamentos and municipios (gov.co datos abiertos).csv")
columns_to_drop = ['REGION', 'CÓDIGO DANE DEL DEPARTAMENTO', 'CÓDIGO DANE DEL MUNICIPIO']  # Replace with the names of the columns you want to drop
departments = departments.drop(columns=columns_to_drop)
print(departments['DEPARTAMENTO'].unique())


departments["DEPARTAMENTO"] = departments["DEPARTAMENTO"].astype(str)
df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
df_2012["City"] = df_2012["City"].astype(str)
import unicodedata
def remove_accents(input_str):
    normalized_str = unicodedata.normalize('NFD', input_str)
    return ''.join(c for c in normalized_str if unicodedata.category(c) != 'Mn').lower()

departments['Cleaned Department'] = departments['DEPARTAMENTO'].apply(remove_accents)
departments['Cleaned Municipio'] = departments['MUNICIPIO'].apply(remove_accents)
df_2012['Administrative Region'] = df_2012['Administrative Region'].apply(remove_accents)
df_2012['City'] = df_2012['City'].apply(remove_accents)

df_2012['Department'] = None

['Antioquia' 'Boyacá' 'Córdoba' 'Chocó' 'Nariño' 'Santander' 'Meta'
 'Atlántico' 'Bolívar' 'Caldas' 'Caquetá' 'Cauca' 'Cesar' 'Cundinamarca'
 'Huila' 'La Guajira' 'Magdalena' 'Quindío' 'Risaralda' 'Sucre' 'Tolima'
 'Arauca' 'Casanare' 'Putumayo' 'Amazonas' 'Guainía' 'Vaupés' 'Vichada'
 'Guaviare' 'Archipiélago de San Andrés, Providencia y Santa Catalina'
 'Bogotá D.C.' 'Norte de Santander' 'Valle del Cauca']


<ipython-input-187-9cf0af21eb4e>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["Administrative Region"] = df_2012["Administrative Region"].astype(str)
<ipython-input-187-9cf0af21eb4e>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2012["City"] = df_2012["City"].astype(str)
<ipython-input-187-9cf0af21eb4e>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentati

In [ ]:
for index, row in df_2012.iterrows():
  admin_region = row['Administrative Region']
  matching_dept = departments.loc[departments['Cleaned Department'] == admin_region, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
270339,2022-12-30,COL,Cocaine base,1.8,kg,cartagena del chaira,caqueta,Caquetá
270340,2022-12-29,COL,"Cocaine hydrochloride (HCl, powder cocaine)",2.596,kg,san andres,san andres islas,None
270341,2022-12-28,COL,Cocaine base,227.22,kg,caloto,cauca,Cauca
270342,2022-12-28,COL,Cocaine base,12.0,kg,el tambo,cauca,Cauca
270343,2022-12-28,COL,Cocaine base,18.025,kg,el tambo,cauca,Cauca


In [ ]:
for index, row in df_2012.iterrows():
  city = row['City']
  matching_dept = departments.loc[departments['Cleaned Municipio'] == city, 'DEPARTAMENTO']
  if not matching_dept.empty:
    df_2012.at[index, 'Department'] = matching_dept.values[0]

df_2012.head()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
270339,2022-12-30,COL,Cocaine base,1.8,kg,cartagena del chaira,caqueta,Caquetá
270340,2022-12-29,COL,"Cocaine hydrochloride (HCl, powder cocaine)",2.596,kg,san andres,san andres islas,Santander
270341,2022-12-28,COL,Cocaine base,227.22,kg,caloto,cauca,Cauca
270342,2022-12-28,COL,Cocaine base,12.0,kg,el tambo,cauca,Cauca
270343,2022-12-28,COL,Cocaine base,18.025,kg,el tambo,cauca,Cauca


In [ ]:
none_department_rows = df_2012[df_2012["Department"].isna()]
print(len(none_department_rows))
none_department_rows['City'].unique()

0


array([], dtype=object)

In [ ]:
df_2012.to_csv("/content/drive/MyDrive/AAA/2022.csv", index=False)

###Aggregate and clean

In [ ]:
df2012 = pd.read_csv("/content/drive/MyDrive/AAA/2012.csv")
df2013 = pd.read_csv("/content/drive/MyDrive/AAA/2013.csv")
df2014 = pd.read_csv("/content/drive/MyDrive/AAA/2014.csv")
df2015 = pd.read_csv("/content/drive/MyDrive/AAA/2015.csv")
df2016 = pd.read_csv("/content/drive/MyDrive/AAA/2016.csv")
df2017 = pd.read_csv("/content/drive/MyDrive/AAA/2017.csv")
df2018 = pd.read_csv("/content/drive/MyDrive/AAA/2018.csv")
df2019 = pd.read_csv("/content/drive/MyDrive/AAA/2019.csv")
df2020 = pd.read_csv("/content/drive/MyDrive/AAA/2020.csv")
df2021 = pd.read_csv("/content/drive/MyDrive/AAA/2021.csv")
df2022 = pd.read_csv("/content/drive/MyDrive/AAA/2022.csv")

In [ ]:
df2022 = df2022.sort_values(by=['Department', 'Seizure Date']).reset_index(drop=True)

In [ ]:
df2022.tail()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
2839,2022-10-24,COL,Cocaine base,37.870,kg,cumaribo,vichada,Vichada
2840,2022-11-04,COL,Cocaine base,27.070,kg,cumaribo,vichada,Vichada
2841,2022-11-04,COL,Cocaine base,32.460,kg,cumaribo,vichada,Vichada
2842,2022-12-09,COL,Cocaine base,24.345,kg,cumaribo,vichada,Vichada
2843,2022-12-09,COL,Cocaine base,59.510,kg,cumaribo,vichada,Vichada


In [ ]:
df_combined = pd.concat([df2012,df2013, df2014, df2015, df2016, df2017, df2018, df2019, df2020, df2021, df2022], ignore_index=True)


In [ ]:
df_combined = df_combined.sort_values(by=['Seizure Date', 'Department']).reset_index(drop=True)


In [ ]:
df_combined.tail()

,Seizure Date,ISO3,Drug/Substance,Quantity Seized,Measurement Unit,City,Administrative Region,Department
268801,2022-12-28,COL,Cocaine base,227.22,kg,caloto,cauca,Cauca
268802,2022-12-28,COL,Cocaine base,12.0,kg,el tambo,cauca,Cauca
268803,2022-12-28,COL,Cocaine base,18.025,kg,el tambo,cauca,Cauca
268804,2022-12-29,COL,"Cocaine hydrochloride (HCl, powder cocaine)",2.596,kg,san andres,san andres islas,Santander
268805,2022-12-30,COL,Cocaine base,1.8,kg,cartagena del chaira,caqueta,Caquetá


In [ ]:
df_combined = df_combined.drop(columns=['ISO3', 'Drug/Substance', 'City', 'Administrative Region']).reset_index(drop=True)

In [ ]:
df_combined.tail()

,Seizure Date,Quantity Seized,Measurement Unit,Department
268801,2022-12-28,227.22,kg,Cauca
268802,2022-12-28,12.0,kg,Cauca
268803,2022-12-28,18.025,kg,Cauca
268804,2022-12-29,2.596,kg,Santander
268805,2022-12-30,1.8,kg,Caquetá


In [ ]:
df_combined['Measurement Unit'].unique()

array(['kg', 'unit', 'lt', 'g'], dtype=object)

In [ ]:
for index, row in df_combined.iterrows():
    if row['Measurement Unit'] == 'g':  # Check if measurement unit is in grams
        df_combined.at[index, 'Quantity Seized'] = float(row['Quantity Seized']) / 1000  # Convert to kilograms
        df_combined.at[index, 'Measurement Unit'] = 'kg'  # Convert to kilograms


In [ ]:
indices = [48788, 49335,107744 ]
new_val = [.364, .6825, 11.375]
new_unit = ['kg', 'kg', 'kg']
df_combined.loc[48788, 'Quantity Seized'] = .364
df_combined.loc[49335, 'Quantity Seized'] = .6825
df_combined.loc[48788, 'Measurement Unit'] = 'kg'
df_combined.loc[49335, 'Measurement Unit'] = 'kg'
df_combined.loc[107744, 'Measurement Unit'] = 'kg'

In [ ]:
df_combined = df_combined.sort_values(by=['Department', 'Seizure Date']).reset_index(drop=True)
df_combined.tail()

,Seizure Date,Quantity Seized,Measurement Unit,Department
268801,2022-10-24,37.87,kg,Vichada
268802,2022-11-04,27.07,kg,Vichada
268803,2022-11-04,32.46,kg,Vichada
268804,2022-12-09,24.345,kg,Vichada
268805,2022-12-09,59.51,kg,Vichada


In [ ]:
df_combined['Seizure Date'] = pd.to_datetime(df_combined['Seizure Date'])
df_combined.head()

,Seizure Date,Quantity Seized,Measurement Unit,Department
0,2012-01-26,0.149,kg,Amazonas
1,2012-03-16,0.100,kg,Amazonas
2,2012-04-26,65.161,kg,Amazonas
3,2012-05-19,5.484,kg,Amazonas
4,2012-05-29,0.160,kg,Amazonas


In [ ]:
df_combined.rename(columns={'Quantity Seized': 'Quantity Seized (kg)'}, inplace=True)
df_combined = df_combined.drop(columns=['Measurement Unit']).reset_index(drop=True)

In [ ]:
df_combined.tail()

,Seizure Date,Quantity Seized (kg),Department
268801,2022-10-24,37.87,Vichada
268802,2022-11-04,27.07,Vichada
268803,2022-11-04,32.46,Vichada
268804,2022-12-09,24.345,Vichada
268805,2022-12-09,59.51,Vichada


In [ ]:
x = df_combined[df_combined['Quantity Seized (kg)'].str.contains(',', na=False)]
x['Quantity Seized (kg)']= pd.to_numeric(x['Quantity Seized (kg)'].str.replace(',', ''), errors='coerce')
x.tail()

<ipython-input-340-1cf18a1950cc>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x['Quantity Seized (kg)']= pd.to_numeric(x['Quantity Seized (kg)'].str.replace(',', ''), errors='coerce')


,Seizure Date,Quantity Seized (kg),Department
261565,2017-10-30,1044.0,Valle del Cauca
261584,2017-11-02,1517.0,Valle del Cauca
261625,2017-11-14,2123.0,Valle del Cauca
261652,2017-11-22,1558.0,Valle del Cauca
261673,2017-11-28,1285.7,Valle del Cauca


In [ ]:
for index, row in df_combined.iterrows():
  if ',' in str(row['Quantity Seized (kg)']):
    df_combined.at[index, 'Quantity Seized (kg)'] = float(row['Quantity Seized (kg)'].replace(',', ''))

In [ ]:
#df_combined['Quantity Seized (kg)'] = pd.to_numeric(df_combined['Quantity Seized (kg)'].str.replace(',', ''), errors='coerce')

df_combined.head()

,Seizure Date,Quantity Seized (kg),Department
0,2012-01-26,0.149,Amazonas
1,2012-03-16,0.100,Amazonas
2,2012-04-26,65.161,Amazonas
3,2012-05-19,5.484,Amazonas
4,2012-05-29,0.160,Amazonas


In [ ]:
df_combined.tail()

,Seizure Date,Quantity Seized (kg),Department
268801,2022-10-24,37.87,Vichada
268802,2022-11-04,27.07,Vichada
268803,2022-11-04,32.46,Vichada
268804,2022-12-09,24.345,Vichada
268805,2022-12-09,59.51,Vichada


In [ ]:
df_combined.sort_values(by=['Department', 'Seizure Date'], inplace=True)
df_combined.tail()

,Seizure Date,Quantity Seized (kg),Department
268801,2022-10-24,37.87,Vichada
268802,2022-11-04,27.07,Vichada
268803,2022-11-04,32.46,Vichada
268804,2022-12-09,24.345,Vichada
268805,2022-12-09,59.51,Vichada


In [ ]:
df_combined['Department'] = df_combined['Department'].astype(str)
df_combined['Quantity Seized (kg)'] = df_combined['Quantity Seized (kg)'].astype(float)

In [ ]:
df_combined.tail()

,Seizure Date,Quantity Seized (kg),Department
268801,2022-10-24,37.870,Vichada
268802,2022-11-04,27.070,Vichada
268803,2022-11-04,32.460,Vichada
268804,2022-12-09,24.345,Vichada
268805,2022-12-09,59.510,Vichada


In [ ]:

start_date = df_combined['Seizure Date'].min()
end_date = df_combined['Seizure Date'].max()
all_dates = pd.date_range(start=start_date, end=end_date)
#@CITATION: CODE ADAPTED FROM CHAT GPT
#Get unique departments
departments = df_combined['Department'].unique()

# Create an empty list to store all records
all_records = []

# Create rows for each department and each date
for dept in departments:
    for date in all_dates:
        all_records.append({
            'Department': dept,
            'Seizure Date': date,
            'Quantity Seized (kg)': 0,
            'Year': date.year
        })

# Convert the list of records into a DataFrame
all_records_df = pd.DataFrame(all_records)

# Merge the original DataFrame with the new DataFrame of all records
final_df = pd.merge(all_records_df, df_combined, on=['Department', 'Seizure Date'], how='left', suffixes=('', '_original'))

# Fill missing quantities with the original quantities where available
final_df['Quantity Seized (kg)'] = final_df['Quantity Seized (kg)_original'].combine_first(final_df['Quantity Seized (kg)'])

# Drop the original quantity column used for merging
final_df.drop(columns=['Quantity Seized (kg)_original'], inplace=True)

# Sort the final DataFrame by department and date for better readability
final_df = final_df.sort_values(by=['Department', 'Seizure Date']).reset_index(drop=True)

final_df.head(15)

#Now we need to append any cases where multiple drug seizures in one day
# Group by 'Department' and 'Seizure Date' and aggregate quantities
aggregated_df = final_df.groupby(['Department', 'Seizure Date'], as_index=False)['Quantity Seized (kg)'].sum()

# Reset index if necessary
aggregated_df = aggregated_df.reset_index(drop=True)

aggregated_df.head(50)

,Department,Seizure Date,Quantity Seized (kg)
0,Amazonas,2012-01-01,0.000
1,Amazonas,2012-01-02,0.000
2,Amazonas,2012-01-03,0.000
3,Amazonas,2012-01-04,0.000
4,Amazonas,2012-01-05,0.000
5,Amazonas,2012-01-06,0.000
6,Amazonas,2012-01-07,0.000
7,Amazonas,2012-01-08,0.000
8,Amazonas,2012-01-09,0.000
9,Amazonas,2012-01-10,0.000


In [ ]:
aggregated_df.tail()

,Department,Seizure Date,Quantity Seized (kg)
132556,Vichada,2022-12-26,0.0
132557,Vichada,2022-12-27,0.0
132558,Vichada,2022-12-28,0.0
132559,Vichada,2022-12-29,0.0
132560,Vichada,2022-12-30,0.0


In [ ]:
aggregated_df.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/aggregated_11_12.csv", index=False)

###Combine features and drug seizures

In [ ]:
gdl_train= pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/Prev/GDL_data_train.csv")
gdl_test = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/Prev/GDL_data_test.csv")

In [ ]:
gdl_train.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2'], inplace=True)
gdl_test.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2'], inplace=True)

In [ ]:
gdl_train.head()

,Department,Date,Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Income Index,Health Index,Educational Index,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density
0,Amazonas,2012-01-01,0.376,0.676,0.950,58.3,24.4,0.624,0.823,0.602,7.017,8.734,73.49,0.578225
1,Antioquia,2012-01-01,0.763,0.746,0.995,81.6,14.0,0.742,0.865,0.645,7.610,9.520,76.24,0.578225
2,Arauca,2012-01-01,0.771,0.728,1.001,78.4,11.6,0.726,0.876,0.606,7.046,9.410,76.91,0.578225
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.668,0.768,0.992,82.5,14.1,0.747,0.864,0.703,9.582,9.549,76.14,0.578225
4,Atlántico,2012-01-01,0.945,0.752,0.981,80.3,18.4,0.736,0.852,0.679,8.453,9.477,75.36,0.578225


In [ ]:
gdl_data = pd.concat([gdl_train, gdl_test], ignore_index=True)
gdl_data.tail()

,Department,Date,Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Income Index,Health Index,Educational Index,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density
138562,Sucre,2023-06-30,0.5460,NaN,NaN,91.7,0.01,NaN,NaN,NaN,NaN,NaN,NaN,1.232261
138563,Tolima,2023-06-30,0.6870,NaN,NaN,84.8,0.01,NaN,NaN,NaN,NaN,NaN,NaN,1.232261
138564,Valle del Cauca,2023-06-30,0.9020,NaN,NaN,99.3,5.38,NaN,NaN,NaN,NaN,NaN,NaN,1.232261
138565,Vaupés,2023-06-30,0.0001,NaN,NaN,24.2,78.50,NaN,NaN,NaN,NaN,NaN,NaN,1.232261
138566,Vichada,2023-06-30,0.9990,NaN,NaN,99.9,1.46,NaN,NaN,NaN,NaN,NaN,NaN,1.232261


In [ ]:
gdl_data['Date'] = pd.to_datetime(gdl_data['Date'], errors='coerce')

# Filter the DataFrame to keep rows with dates after or in 2023
gdl_data = gdl_data[gdl_data['Date'].dt.year < 2023]


In [ ]:
gdl_data.drop(columns=['Income Index', 'Health Index', 'Educational Index'], inplace=True)

In [ ]:
gdl_data.head()

,Department,Date,Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density
0,Amazonas,2012-01-01,0.376,0.676,0.950,58.3,24.4,7.017,8.734,73.49,0.578225
1,Antioquia,2012-01-01,0.763,0.746,0.995,81.6,14.0,7.610,9.520,76.24,0.578225
2,Arauca,2012-01-01,0.771,0.728,1.001,78.4,11.6,7.046,9.410,76.91,0.578225
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.668,0.768,0.992,82.5,14.1,9.582,9.549,76.14,0.578225
4,Atlántico,2012-01-01,0.945,0.752,0.981,80.3,18.4,8.453,9.477,75.36,0.578225


In [ ]:
drug_seizures = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/aggregated_11_12.csv")
drug_seizures.rename(columns={'Seizure Date': 'Date'}, inplace=True)
drug_seizures.head()

,Department,Date,Quantity Seized (kg)
0,Amazonas,2012-01-01,0.0
1,Amazonas,2012-01-02,0.0
2,Amazonas,2012-01-03,0.0
3,Amazonas,2012-01-04,0.0
4,Amazonas,2012-01-05,0.0


In [ ]:
gdl_data['Date'] = pd.to_datetime(gdl_data['Date'], format='%Y-%m-%d')
drug_seizures['Date'] = pd.to_datetime(drug_seizures['Date'], format='%Y-%m-%d')
gdl_data = gdl_data.sort_values(by=['Date', 'Department']).reset_index(drop=True)
drug_seizures = drug_seizures.sort_values(by=['Date', 'Department']).reset_index(drop=True)

In [ ]:
data = pd.merge(drug_seizures, gdl_data, on=['Department', 'Date'], how='left')

In [ ]:
data.head()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density
0,Amazonas,2012-01-01,0.000,0.376,0.676,0.950,58.3,24.4,7.017,8.734,73.49,0.578225
1,Antioquia,2012-01-01,0.123,0.763,0.746,0.995,81.6,14.0,7.610,9.520,76.24,0.578225
2,Arauca,2012-01-01,0.000,0.771,0.728,1.001,78.4,11.6,7.046,9.410,76.91,0.578225
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.000,0.668,0.768,0.992,82.5,14.1,9.582,9.549,76.14,0.578225
4,Atlántico,2012-01-01,0.000,0.945,0.752,0.981,80.3,18.4,8.453,9.477,75.36,0.578225


In [ ]:
#corruption, homicides, gini, poorest household, border, ocean

In [ ]:
train = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/train_12-19.csv")
test = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/test_20-22.csv")
combined_extra = pd.concat([train, test], ignore_index=True)

combined_extra.head()

,Unnamed: 0,Date,Department,Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Income Index,Health Index,...,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Quantity Seized (kg),Gini Coefficient,% Poorest Household (IWI under 30)
0,0,2012-01-01,Amazonas,0.376,0.676,0.950,58.3,24.4,0.624,0.823,...,7.017,8.734,73.49,0.578225,1.0,0.019231,10,0.000,0.13,0.17
1,1,2012-01-01,Antioquia,0.763,0.746,0.995,81.6,14.0,0.742,0.865,...,7.610,9.520,76.24,0.578225,3.0,0.057692,3083,0.123,0.13,0.17
2,2,2012-01-01,Arauca,0.771,0.728,1.001,78.4,11.6,0.726,0.876,...,7.046,9.410,76.91,96.082389,0.0,0.000000,155,0.000,0.13,0.17
3,3,2012-01-01,"Archipiélago de San Andrés, Providencia y Sant...",0.668,0.768,0.992,82.5,14.1,0.747,0.864,...,9.582,9.549,76.14,8.658131,0.0,0.000000,16,0.000,0.13,0.17
4,4,2012-01-01,Atlántico,0.945,0.752,0.981,80.3,18.4,0.736,0.852,...,8.453,9.477,75.36,1215.422292,2.0,0.038462,549,0.000,0.13,0.17


In [ ]:
data['Number of Corruption Cases'] = combined_extra['Number of Corruption Cases']
data['Percent of Total Corruption Cases for That Year'] = combined_extra['Percent of Total Corruption Cases for That Year']
data['Number of Homicides that Year'] = combined_extra['Number of Homicides that Year']
data['Gini Coefficient'] = combined_extra['Gini Coefficient']
data['% Poorest Household (IWI under 30)'] = combined_extra['% Poorest Household (IWI under 30)']
data.head()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Gini Coefficient,% Poorest Household (IWI under 30)
0,Amazonas,2012-01-01,0.000,0.376,0.676,0.950,58.3,24.4,7.017,8.734,73.49,0.578225,1.0,0.019231,10,0.13,0.17
1,Antioquia,2012-01-01,0.123,0.763,0.746,0.995,81.6,14.0,7.610,9.520,76.24,0.578225,3.0,0.057692,3083,0.13,0.17
2,Arauca,2012-01-01,0.000,0.771,0.728,1.001,78.4,11.6,7.046,9.410,76.91,0.578225,0.0,0.000000,155,0.13,0.17
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.000,0.668,0.768,0.992,82.5,14.1,9.582,9.549,76.14,0.578225,0.0,0.000000,16,0.13,0.17
4,Atlántico,2012-01-01,0.000,0.945,0.752,0.981,80.3,18.4,8.453,9.477,75.36,0.578225,2.0,0.038462,549,0.13,0.17


In [ ]:
data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/data_11_12.csv", index=False)

###Days standardized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/data_11_12.csv")
from sklearn.preprocessing import StandardScaler

In [ ]:
data.head()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Gini Coefficient,% Poorest Household (IWI under 30)
0,Amazonas,2012-01-01,0.000,0.376,0.676,0.950,58.3,24.4,7.017,8.734,73.49,0.578225,1.0,0.019231,10,0.13,0.17
1,Antioquia,2012-01-01,0.123,0.763,0.746,0.995,81.6,14.0,7.610,9.520,76.24,0.578225,3.0,0.057692,3083,0.13,0.17
2,Arauca,2012-01-01,0.000,0.771,0.728,1.001,78.4,11.6,7.046,9.410,76.91,0.578225,0.0,0.000000,155,0.13,0.17
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.000,0.668,0.768,0.992,82.5,14.1,9.582,9.549,76.14,0.578225,0.0,0.000000,16,0.13,0.17
4,Atlántico,2012-01-01,0.000,0.945,0.752,0.981,80.3,18.4,8.453,9.477,75.36,0.578225,2.0,0.038462,549,0.13,0.17


In [ ]:
columns = [
       'Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Gini Coefficient',
       '% Poorest Household (IWI under 30)']

features = ['Department', 'Date', 'Quantity Seized (kg)']
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')

train = data[data['Date'].dt.year < 2020]
test = data[data['Date'].dt.year >= 2020].reset_index(drop=True)


X_train = train[columns]
X_test = test[columns]

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled= scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=columns)

train_standardized = pd.concat([train[features], X_train_scaled], axis=1)
test_standardized = pd.concat([test[features], X_test_scaled], axis=1)


In [ ]:
data_standardized = pd.concat([train_standardized, test_standardized], ignore_index=True)
data_standardized.head()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Gini Coefficient,% Poorest Household (IWI under 30)
0,Amazonas,2012-01-01,0.000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,0.103041,1.095493
1,Antioquia,2012-01-01,0.123,0.462456,0.295543,0.280588,0.359676,-0.366227,0.058038,0.552396,0.164933,-0.324089,0.199369,0.644118,4.536211,0.103041,1.095493
2,Arauca,2012-01-01,0.000,0.505368,-0.203322,0.614262,0.073946,-0.599340,-0.502880,0.230755,0.531607,-0.324089,-0.683264,-0.712641,-0.416973,0.103041,1.095493
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.000,-0.047116,0.905267,0.113752,0.440038,-0.356514,2.019261,0.637193,0.110205,-0.324089,-0.683264,-0.712641,-0.652114,0.103041,1.095493
4,Atlántico,2012-01-01,0.000,1.438689,0.461831,-0.497983,0.243598,0.061146,0.896431,0.426664,-0.316669,-0.324089,-0.094842,0.191865,0.249541,0.103041,1.095493


In [ ]:
border_depts = ['Nariño', 'Putumayo', 'Amazonas', 'Vaupés', 'Guainia', 'Vichada', 'Arauca', 'Norte de Santander', 'Cesar', 'La Guajira', 'Choco' ]
ocean_depts = ['Nariño', 'Cauca', 'Valle del Cauca', 'Chocó', 'Antioquia', 'Córdoba', 'Sucre', 'Bolívar', 'Atlántico', 'Magdalena', 'La Guajira']
data_standardized['Border_Department'] = data_standardized['Department'].apply(lambda x: 1 if x in border_depts else 0)
data_standardized['Ocean_department'] = data_standardized['Department'].apply(lambda x: 1 if x in ocean_depts else 0)

data_standardized.drop(columns=['Gini Coefficient', '% Poorest Household (IWI under 30)'], inplace=True)

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
data_standardized.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_standardized_11_12.csv", index=False)

###Days normalized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/data_11_12.csv")
from sklearn.preprocessing import MinMaxScaler

In [ ]:
columns = [
       'Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Gini Coefficient',
       '% Poorest Household (IWI under 30)']

features = ['Department', 'Date', 'Quantity Seized (kg)']
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')

train = data[data['Date'].dt.year < 2020]
test = data[data['Date'].dt.year >= 2020].reset_index(drop=True)


X_train = train[columns]
X_test = test[columns]

scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled= scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=columns)

train_standardized = pd.concat([train[features], X_train_scaled], axis=1)
test_standardized = pd.concat([test[features], X_test_scaled], axis=1)


In [ ]:
data_standardized = pd.concat([train_standardized, test_standardized], ignore_index=True)
data_standardized.head()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Gini Coefficient,% Poorest Household (IWI under 30)
0,Amazonas,2012-01-01,0.000,0.241798,0.241758,0.316327,0.408428,0.388707,0.246288,0.343883,0.408556,0.000018,0.037037,0.056561,0.002807,0.257143,0.269414
1,Antioquia,2012-01-01,0.123,0.712029,0.626374,0.775510,0.786062,0.209118,0.372083,0.822276,0.702674,0.000018,0.111111,0.169683,0.865525,0.257143,0.269414
2,Arauca,2012-01-01,0.000,0.721750,0.527473,0.836735,0.734198,0.167674,0.252440,0.755326,0.774332,0.000018,0.000000,0.000000,0.043515,0.257143,0.269414
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.000,0.596598,0.747253,0.744898,0.800648,0.210844,0.790412,0.839927,0.691979,0.000018,0.000000,0.000000,0.004492,0.257143,0.269414
4,Atlántico,2012-01-01,0.000,0.933171,0.659341,0.632653,0.764992,0.285098,0.550912,0.796105,0.608556,0.000018,0.074074,0.113122,0.154127,0.257143,0.269414


In [ ]:
border_depts = ['Nariño', 'Putumayo', 'Amazonas', 'Vaupés', 'Guainia', 'Vichada', 'Arauca', 'Norte de Santander', 'Cesar', 'La Guajira', 'Choco' ]
ocean_depts = ['Nariño', 'Cauca', 'Valle del Cauca', 'Chocó', 'Antioquia', 'Córdoba', 'Sucre', 'Bolívar', 'Atlántico', 'Magdalena', 'La Guajira']
data_standardized['Border_Department'] = data_standardized['Department'].apply(lambda x: 1 if x in border_depts else 0)
data_standardized['Ocean_department'] = data_standardized['Department'].apply(lambda x: 1 if x in ocean_depts else 0)

data_standardized.drop(columns=['Gini Coefficient', '% Poorest Household (IWI under 30)'], inplace=True)

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
data_standardized.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_normalized_11_12.csv", index=False)

###Months standardized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_standardized_11_12.csv")

In [ ]:
data.head()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,Amazonas,2012-01-01,0.000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,1,0
1,Antioquia,2012-01-01,0.123,0.462456,0.295543,0.280588,0.359676,-0.366227,0.058038,0.552396,0.164933,-0.324089,0.199369,0.644118,4.536211,0,1
2,Arauca,2012-01-01,0.000,0.505368,-0.203322,0.614262,0.073946,-0.599340,-0.502880,0.230755,0.531607,-0.324089,-0.683264,-0.712641,-0.416973,1,0
3,"Archipiélago de San Andrés, Providencia y Sant...",2012-01-01,0.000,-0.047116,0.905267,0.113752,0.440038,-0.356514,2.019261,0.637193,0.110205,-0.324089,-0.683264,-0.712641,-0.652114,0,0
4,Atlántico,2012-01-01,0.000,1.438689,0.461831,-0.497983,0.243598,0.061146,0.896431,0.426664,-0.316669,-0.324089,-0.094842,0.191865,0.249541,0,1


In [ ]:
data['Date'] = pd.to_datetime(data['Date'])
data['month_year'] = data['Date'].dt.to_period('M')


In [ ]:
columns_to_aggregate = ['Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department', 'Ocean_department']

# Group by month_year and Department, summing the quantity seized and taking the first for other columns
month_data = (data
                  .groupby(['month_year', 'Department'])
                  .agg({'Quantity Seized (kg)': 'sum',
                        **{col: 'first' for col in columns_to_aggregate}})
                  .reset_index())
month_data.rename(columns={'Quantity Seized (kg)': 'Monthly Quantity Seized (kg)'}, inplace=True)


In [ ]:
month_data['Log Monthly Quantity Seized (kg)'] = np.log1p(month_data['Monthly Quantity Seized (kg)'])


In [ ]:
month_data = month_data.sort_values(by=['Department', 'month_year'])

In [ ]:
month_data['month_year'] = month_data['month_year'].dt.to_timestamp()

# Define the starting date (January 2012 as month 0)
start_date = pd.to_datetime('2012-01')

# Calculate month number by finding the difference in months
month_data['month_number'] = (month_data['month_year'].dt.year - start_date.year) * 12 + (month_data['month_year'].dt.month - start_date.month)

In [ ]:
month_data = month_data.sort_values(by=['Department', 'month_number']).reset_index(drop=True)

In [ ]:
cols_to_move = ['month_number', 'Department', 'Monthly Quantity Seized (kg)', 'Log Monthly Quantity Seized (kg)']
cols = cols_to_move + [col for col in month_data.columns if col not in cols_to_move]
month_data = month_data[cols]

month_data.drop(columns=['month_year'], inplace=True)

<ipython-input-249-edbb496f9e03>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  month_data.drop(columns=['month_year'], inplace=True)


In [ ]:
month_data.head()

,month_number,Department,Monthly Quantity Seized (kg),Log Monthly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.149,0.138892,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,1,0
1,1,Amazonas,0.000,0.000000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.313802,-0.389053,-0.260388,-0.662264,1,0
2,2,Amazonas,0.100,0.095310,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,5.234639,-0.389053,-0.260388,-0.662264,1,0
3,3,Amazonas,65.161,4.192091,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.158489,-0.389053,-0.260388,-0.662264,1,0
4,4,Amazonas,6.344,1.993884,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.314416,-0.389053,-0.260388,-0.662264,1,0


In [ ]:

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
month_data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/month_data_standardized.csv", index=False)

###Months standardized exploration

In [ ]:
months = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/month_data_standardized.csv")
months.head()

,month_number,Department,Monthly Quantity Seized (kg),Log Monthly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.149,0.138892,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,1,0
1,1,Amazonas,0.000,0.000000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.313802,-0.389053,-0.260388,-0.662264,1,0
2,2,Amazonas,0.100,0.095310,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,5.234639,-0.389053,-0.260388,-0.662264,1,0
3,3,Amazonas,65.161,4.192091,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.158489,-0.389053,-0.260388,-0.662264,1,0
4,4,Amazonas,6.344,1.993884,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.314416,-0.389053,-0.260388,-0.662264,1,0


In [ ]:
train = months[months['month_number'] <= 107]

In [ ]:
train.tail()

,month_number,Department,Monthly Quantity Seized (kg),Log Monthly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
4327,103,Vichada,0.000,0.000000,1.728341,0.212399,-0.275534,1.341875,-1.115101,1.099316,0.239527,-0.365924,-0.266563,-0.683264,-0.712641,-0.650422,1,0
4328,104,Vichada,0.000,0.000000,1.728341,0.212399,-0.275534,1.341875,-1.115101,1.099316,0.239527,-0.365924,-0.257800,-0.683264,-0.712641,-0.650422,1,0
4329,105,Vichada,0.000,0.000000,1.728341,0.212399,-0.275534,1.341875,-1.115101,1.099316,0.239527,-0.365924,-0.307077,-0.683264,-0.712641,-0.650422,1,0
4330,106,Vichada,35.165,3.588092,1.728341,0.212399,-0.275534,1.341875,-1.115101,1.099316,0.239527,-0.365924,-0.230935,-0.683264,-0.712641,-0.650422,1,0
4331,107,Vichada,0.000,0.000000,1.728341,0.212399,-0.275534,1.341875,-1.115101,1.099316,0.239527,-0.365924,-0.045086,-0.683264,-0.712641,-0.650422,1,0


In [ ]:
import pandas as pd
#@CITATION: CODE ADAPTED FROM CHAT GPT
# Summary statistics for Monthly Quantity Seized (kg)
summary_stats = train['Monthly Quantity Seized (kg)'].describe()

# Compute additional statistics
iqr = summary_stats['75%'] - summary_stats['25%']  # Interquartile Range
std_dev = summary_stats['std']  # Standard Deviation
cv = std_dev / summary_stats['mean']  # Coefficient of Variation

# Create a nicely formatted DataFrame
summary_df = pd.DataFrame({
    'Statistic': ['Count', 'Mean', 'Standard Deviation', 'Min', '25th Percentile (Q1)',
                  'Median (Q2)', '75th Percentile (Q3)', 'Max', 'Interquartile Range (IQR)',
                  'Coefficient of Variation (CV)'],
    'Value': [summary_stats['count'], summary_stats['mean'], summary_stats['std'],
              summary_stats['min'], summary_stats['25%'], summary_stats['50%'],
              summary_stats['75%'], summary_stats['max'], iqr, cv]
})

# Print the table
print(summary_df.to_string(index=False))


                    Statistic        Value
                        Count  3564.000000
                         Mean   651.298978
           Standard Deviation  1619.535729
                          Min     0.000000
         25th Percentile (Q1)     1.278725
                  Median (Q2)    58.488400
         75th Percentile (Q3)   537.743750
                          Max 21562.736000
    Interquartile Range (IQR)   536.465025
Coefficient of Variation (CV)     2.486624


###Months normalized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_normalized_11_12.csv")


In [ ]:
data['Date'] = pd.to_datetime(data['Date'])
data['month_year'] = data['Date'].dt.to_period('M')

columns_to_aggregate = ['Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department', 'Ocean_department']
#@CITATION: CODE ADAPTED FROM CHAT GPT
# Group by month_year and Department, summing the quantity seized and taking the first for other columns
month_data = (data
                  .groupby(['month_year', 'Department'])
                  .agg({'Quantity Seized (kg)': 'sum',
                        **{col: 'first' for col in columns_to_aggregate}})
                  .reset_index())
month_data.rename(columns={'Quantity Seized (kg)': 'Monthly Quantity Seized (kg)'}, inplace=True)

month_data['Log Monthly Quantity Seized (kg)'] = np.log1p(month_data['Monthly Quantity Seized (kg)'])

month_data = month_data.sort_values(by=['Department', 'month_year'])

month_data['month_year'] = month_data['month_year'].dt.to_timestamp()

# Define the starting date (January 2012 as month 0)
start_date = pd.to_datetime('2012-01')

# Calculate month number by finding the difference in months
month_data['month_number'] = (month_data['month_year'].dt.year - start_date.year) * 12 + (month_data['month_year'].dt.month - start_date.month)

month_data = month_data.sort_values(by=['Department', 'month_number']).reset_index(drop=True)

cols_to_move = ['month_number', 'Department', 'Monthly Quantity Seized (kg)', 'Log Monthly Quantity Seized (kg)']
cols = cols_to_move + [col for col in month_data.columns if col not in cols_to_move]
month_data = month_data[cols]

month_data.drop(columns=['month_year'], inplace=True)


In [ ]:
month_data.tail()

,month_number,Department,Monthly Quantity Seized (kg),Log Monthly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
4351,127,Vichada,810.000,6.698268,0.998785,0.620879,0.77551,1.04376,0.020376,0.594188,0.833841,0.479144,0.010444,0.0,0.0,0.005053,1,0
4352,128,Vichada,0.000,0.000000,0.998785,0.620879,0.77551,1.04376,0.020376,0.594188,0.833841,0.479144,0.011459,0.0,0.0,0.005053,1,0
4353,129,Vichada,537.870,6.289474,0.998785,0.620879,0.77551,1.04376,0.020376,0.594188,0.833841,0.479144,0.003022,0.0,0.0,0.005053,1,0
4354,130,Vichada,59.530,4.103139,0.998785,0.620879,0.77551,1.04376,0.020376,0.594188,0.833841,0.479144,0.016317,0.0,0.0,0.005053,1,0
4355,131,Vichada,83.855,4.440944,0.998785,0.620879,0.77551,1.04376,0.020376,0.594188,0.833841,0.479144,0.048051,0.0,0.0,0.005053,1,0


In [ ]:

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
month_data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/month_data_normalized.csv", index=False)

###Week Data standardized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_standardized_11_12.csv")


In [ ]:

data['Date'] = pd.to_datetime(data['Date'])

# Sort by date in case it's not sorted
data = data.sort_values(by='Date')

# Calculate the "week number" from the start of your dataset
data['week_number'] = (data['Date'] - data['Date'].min()).dt.days // 7

# Now you can group by department and week_number for aggregation
week_data = data.sort_values(by=['Department', 'week_number'])
week_data['week_number'] = week_data['week_number'].astype(int)
week_data.tail()

,Department,Date,Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department,week_number
132428,Vichada,2022-12-26,0.0,1.728341,0.267828,0.280588,1.779399,-1.42786,1.099316,0.607953,-0.978872,-0.323296,-0.683264,-0.712641,-0.648731,1,0,573
132461,Vichada,2022-12-27,0.0,1.728341,0.267828,0.280588,1.779399,-1.42786,1.099316,0.607953,-0.978872,-0.323296,-0.683264,-0.712641,-0.648731,1,0,573
132494,Vichada,2022-12-28,0.0,1.728341,0.267828,0.280588,1.779399,-1.42786,1.099316,0.607953,-0.978872,-0.323296,-0.683264,-0.712641,-0.648731,1,0,573
132527,Vichada,2022-12-29,0.0,1.728341,0.267828,0.280588,1.779399,-1.42786,1.099316,0.607953,-0.978872,-0.323296,-0.683264,-0.712641,-0.648731,1,0,573
132560,Vichada,2022-12-30,0.0,1.728341,0.267828,0.280588,1.779399,-1.42786,1.099316,0.607953,-0.978872,-0.323296,-0.683264,-0.712641,-0.648731,1,0,573


In [ ]:
week_data.columns

Index(['Department', 'Date', 'Quantity Seized (kg)',
       'Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department',
       'Ocean_department', 'week_number'],
      dtype='object')

In [ ]:

columns_to_aggregate = ['Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department',
       'Ocean_department']

week_data  = (week_data.groupby(['week_number', 'Department']).agg({'Quantity Seized (kg)': 'sum',
                        **{col: 'first' for col in columns_to_aggregate}})
                  .reset_index())
week_data['Log Weekly Quantity Seized (kg)'] = np.log(week_data['Quantity Seized (kg)'])

week_data['Log Weekly Quantity Seized (kg)'] = week_data['Log Weekly Quantity Seized (kg)'].clip(lower=0)
week_data.rename(columns={'Quantity Seized (kg)': 'Weekly Quantity Seized (kg)'}, inplace=True)


cols_to_move = ['week_number', 'Department', 'Weekly Quantity Seized (kg)', 'Log Weekly Quantity Seized (kg)']
cols = cols_to_move + [col for col in week_data.columns if col not in cols_to_move]
week_data = week_data[cols]

week_data.head()



/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,week_number,Department,Weekly Quantity Seized (kg),Log Weekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.000,0.000000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,1,0
1,0,Antioquia,28.665,3.355677,0.462456,0.295543,0.280588,0.359676,-0.366227,0.058038,0.552396,0.164933,-0.324089,0.199369,0.644118,4.536211,0,1
2,0,Arauca,0.000,0.000000,0.505368,-0.203322,0.614262,0.073946,-0.599340,-0.502880,0.230755,0.531607,-0.324089,-0.683264,-0.712641,-0.416973,1,0
3,0,"Archipiélago de San Andrés, Providencia y Sant...",0.000,0.000000,-0.047116,0.905267,0.113752,0.440038,-0.356514,2.019261,0.637193,0.110205,-0.324089,-0.683264,-0.712641,-0.652114,0,0
4,0,Atlántico,0.606,0.000000,1.438689,0.461831,-0.497983,0.243598,0.061146,0.896431,0.426664,-0.316669,-0.324089,-0.094842,0.191865,0.249541,0,1


In [ ]:

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
week_data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/week_data_standardized.csv", index=False)

###Week data normalized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_normalized_11_12.csv")


data['Date'] = pd.to_datetime(data['Date'])
#@CITATION: CODE ADAPTED FROM CHAT GPT
# Sort by date in case it's not sorted
data = data.sort_values(by='Date')

# Calculate the "week number" from the start of your dataset
data['week_number'] = (data['Date'] - data['Date'].min()).dt.days // 7

# Now you can group by department and week_number for aggregation
week_data = data.sort_values(by=['Department', 'week_number'])
week_data['week_number'] = week_data['week_number'].astype(int)
week_data.tail()


columns_to_aggregate = ['Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department', 'Ocean_department']

week_data  = (week_data.groupby(['week_number', 'Department']).agg({'Quantity Seized (kg)': 'sum',
                        **{col: 'first' for col in columns_to_aggregate}})
                  .reset_index())
week_data['Log Weekly Quantity Seized (kg)'] = np.log(week_data['Quantity Seized (kg)'])

week_data['Log Weekly Quantity Seized (kg)'] = week_data['Log Weekly Quantity Seized (kg)'].clip(lower=0)
week_data.rename(columns={'Quantity Seized (kg)': 'Weekly Quantity Seized (kg)'}, inplace=True)


cols_to_move = ['week_number', 'Department', 'Weekly Quantity Seized (kg)', 'Log Weekly Quantity Seized (kg)']
cols = cols_to_move + [col for col in week_data.columns if col not in cols_to_move]
week_data = week_data[cols]

week_data.head()



/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,week_number,Department,Weekly Quantity Seized (kg),Log Weekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.000,0.000000,0.241798,0.241758,0.316327,0.408428,0.388707,0.246288,0.343883,0.408556,0.000018,0.037037,0.056561,0.002807,1,0
1,0,Antioquia,28.665,3.355677,0.712029,0.626374,0.775510,0.786062,0.209118,0.372083,0.822276,0.702674,0.000018,0.111111,0.169683,0.865525,0,1
2,0,Arauca,0.000,0.000000,0.721750,0.527473,0.836735,0.734198,0.167674,0.252440,0.755326,0.774332,0.000018,0.000000,0.000000,0.043515,1,0
3,0,"Archipiélago de San Andrés, Providencia y Sant...",0.000,0.000000,0.596598,0.747253,0.744898,0.800648,0.210844,0.790412,0.839927,0.691979,0.000018,0.000000,0.000000,0.004492,0,0
4,0,Atlántico,0.606,0.000000,0.933171,0.659341,0.632653,0.764992,0.285098,0.550912,0.796105,0.608556,0.000018,0.074074,0.113122,0.154127,0,1


In [ ]:
week_data.tail()

,week_number,Department,Weekly Quantity Seized (kg),Log Weekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
18937,573,Sucre,0.0,0.000000,0.466586,0.560440,0.806122,0.917342,-0.013469,0.371659,0.730980,0.549733,0.000153,0.074074,0.346021,0.056429,0,1
18938,573,Tolima,0.0,0.000000,0.619684,0.576923,0.948980,0.823339,-0.005871,0.396903,0.782106,0.462032,0.000153,0.037037,0.173010,0.101348,0,0
18939,573,Valle del Cauca,4570.0,8.427268,0.877278,0.824176,0.714286,1.051864,0.072181,0.798473,0.997565,0.520856,0.000153,0.000000,0.000000,0.620719,0,1
18940,573,Vaupés,0.0,0.000000,-0.156258,-0.005495,0.051020,-0.108590,1.241754,0.224650,0.035301,-0.167914,0.000153,0.000000,0.000000,0.001404,1,0
18941,573,Vichada,0.0,0.000000,0.998785,0.620879,0.775510,1.043760,0.020376,0.594188,0.833841,0.479144,0.000153,0.000000,0.000000,0.005053,1,0


In [ ]:

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
week_data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/week_data_normalized.csv", index=False)

###Biweekly standardized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_standardized_11_12.csv")

data['Date'] = pd.to_datetime(data['Date'])
#@CITATION: CODE ADAPTED FROM CHAT GPT
# Sort by date in case it's not sorted
data = data.sort_values(by='Date')

# Calculate the "biweek number" from the start of your dataset
data['biweek_number'] = (data['Date'] - data['Date'].min()).dt.days // 14

# Now you can group by department and biweek_number for aggregation
biweek_data = data.sort_values(by=['Department', 'biweek_number'])
biweek_data['biweek_number'] = biweek_data['biweek_number'].astype(int)


columns_to_aggregate = ['Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department', 'Ocean_department']

biweek_data  = (biweek_data.groupby(['biweek_number', 'Department']).agg({'Quantity Seized (kg)': 'sum',
                        **{col: 'first' for col in columns_to_aggregate}})
                  .reset_index())
biweek_data['Log BiWeekly Quantity Seized (kg)'] = np.log(biweek_data['Quantity Seized (kg)'])

biweek_data['Log BiWeekly Quantity Seized (kg)'] = biweek_data['Log BiWeekly Quantity Seized (kg)'].clip(lower=0)
biweek_data.rename(columns={'Quantity Seized (kg)': 'BiWeekly Quantity Seized (kg)'}, inplace=True)


cols_to_move = ['biweek_number', 'Department', 'BiWeekly Quantity Seized (kg)', 'Log BiWeekly Quantity Seized (kg)']
cols = cols_to_move + [col for col in biweek_data.columns if col not in cols_to_move]
biweek_data = biweek_data[cols]

biweek_data.head()



/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,biweek_number,Department,BiWeekly Quantity Seized (kg),Log BiWeekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.000,0.000000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,1,0
1,0,Antioquia,267.710,5.589904,0.462456,0.295543,0.280588,0.359676,-0.366227,0.058038,0.552396,0.164933,-0.324089,0.199369,0.644118,4.536211,0,1
2,0,Arauca,0.000,0.000000,0.505368,-0.203322,0.614262,0.073946,-0.599340,-0.502880,0.230755,0.531607,-0.324089,-0.683264,-0.712641,-0.416973,1,0
3,0,"Archipiélago de San Andrés, Providencia y Sant...",0.000,0.000000,-0.047116,0.905267,0.113752,0.440038,-0.356514,2.019261,0.637193,0.110205,-0.324089,-0.683264,-0.712641,-0.652114,0,0
4,0,Atlántico,2.914,1.069527,1.438689,0.461831,-0.497983,0.243598,0.061146,0.896431,0.426664,-0.316669,-0.324089,-0.094842,0.191865,0.249541,0,1


In [ ]:
biweek_data.tail()

,biweek_number,Department,BiWeekly Quantity Seized (kg),Log BiWeekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
9466,286,Sucre,0.0,0.000000,-0.621055,-0.037034,0.447425,1.082932,-1.618235,0.056048,0.113795,-0.617671,-0.323736,-0.094842,2.054084,-0.339157,0,1
9467,286,Tolima,0.0,0.000000,0.054799,0.046110,1.225996,0.565045,-1.575498,0.174398,0.359412,-1.066436,-0.323736,-0.389053,0.670721,-0.068491,0,0
9468,286,Valle del Cauca,4969.0,8.510974,1.191949,1.293273,-0.053085,1.824045,-1.136470,2.057053,1.394511,-0.765435,-0.323736,-0.683264,-0.712641,3.061082,0,1
9469,286,Vaupés,0.0,0.000000,-3.370598,-2.891652,-3.667878,-4.569173,5.442159,-0.633164,-3.228348,-4.289887,-0.323736,-0.683264,-0.712641,-0.670722,1,0
9470,286,Vichada,0.0,0.000000,1.728341,0.267828,0.280588,1.779399,-1.427860,1.099316,0.607953,-0.978872,-0.323736,-0.683264,-0.712641,-0.648731,1,0


In [ ]:

na_rows = data_standardized[data_standardized.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [Department, Date, Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
biweek_data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/biweek_data_standardized.csv", index=False)

###Bi week normalized

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/day_data_normalized_11_12.csv")

data['Date'] = pd.to_datetime(data['Date'])
#@CITATION: CODE ADAPTED FROM CHAT GPT
# Sort by date in case it's not sorted
data = data.sort_values(by='Date')

# Calculate the "biweek number" from the start of your dataset
data['biweek_number'] = (data['Date'] - data['Date'].min()).dt.days // 14

# Now you can group by department and biweek_number for aggregation
biweek_data = data.sort_values(by=['Department', 'biweek_number'])
biweek_data['biweek_number'] = biweek_data['biweek_number'].astype(int)


columns_to_aggregate = ['Percent Population in Urban Areas', 'HDI', 'GDI',
       'Mean International Wealth Index', 'Infant Deaths per 1000 Live Births',
       'Mean Years Schooling',
       'Log Gross National Income Per Capita in 1000 USD', 'Life Expectancy',
       'Population Density', 'Number of Corruption Cases',
       'Percent of Total Corruption Cases for That Year',
       'Number of Homicides that Year', 'Border_Department', 'Ocean_department']

biweek_data  = (biweek_data.groupby(['biweek_number', 'Department']).agg({'Quantity Seized (kg)': 'sum',
                        **{col: 'first' for col in columns_to_aggregate}})
                  .reset_index())
biweek_data['Log BiWeekly Quantity Seized (kg)'] = np.log(biweek_data['Quantity Seized (kg)'])

biweek_data['Log BiWeekly Quantity Seized (kg)'] = biweek_data['Log BiWeekly Quantity Seized (kg)'].clip(lower=0)
biweek_data.rename(columns={'Quantity Seized (kg)': 'BiWeekly Quantity Seized (kg)'}, inplace=True)


cols_to_move = ['biweek_number', 'Department', 'BiWeekly Quantity Seized (kg)', 'Log BiWeekly Quantity Seized (kg)']
cols = cols_to_move + [col for col in biweek_data.columns if col not in cols_to_move]
biweek_data = biweek_data[cols]

biweek_data.head()



/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,biweek_number,Department,BiWeekly Quantity Seized (kg),Log BiWeekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.000,0.000000,0.241798,0.241758,0.316327,0.408428,0.388707,0.246288,0.343883,0.408556,0.000018,0.037037,0.056561,0.002807,1,0
1,0,Antioquia,267.710,5.589904,0.712029,0.626374,0.775510,0.786062,0.209118,0.372083,0.822276,0.702674,0.000018,0.111111,0.169683,0.865525,0,1
2,0,Arauca,0.000,0.000000,0.721750,0.527473,0.836735,0.734198,0.167674,0.252440,0.755326,0.774332,0.000018,0.000000,0.000000,0.043515,1,0
3,0,"Archipiélago de San Andrés, Providencia y Sant...",0.000,0.000000,0.596598,0.747253,0.744898,0.800648,0.210844,0.790412,0.839927,0.691979,0.000018,0.000000,0.000000,0.004492,0,0
4,0,Atlántico,2.914,1.069527,0.933171,0.659341,0.632653,0.764992,0.285098,0.550912,0.796105,0.608556,0.000018,0.074074,0.113122,0.154127,0,1


In [ ]:

na_rows = biweek_data[biweek_data.isna().any(axis=1)]
print(na_rows)

Empty DataFrame
Columns: [biweek_number, Department, BiWeekly Quantity Seized (kg), Log BiWeekly Quantity Seized (kg), Percent Population in Urban Areas, HDI, GDI, Mean International Wealth Index, Infant Deaths per 1000 Live Births, Mean Years Schooling, Log Gross National Income Per Capita in 1000 USD, Life Expectancy, Population Density, Number of Corruption Cases, Percent of Total Corruption Cases for That Year, Number of Homicides that Year, Border_Department, Ocean_department]
Index: []


In [ ]:
biweek_data.to_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/biweek_data_normalized.csv", index=False)

In [ ]:
biweek_data.tail()

,biweek_number,Department,BiWeekly Quantity Seized (kg),Log BiWeekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
9466,286,Sucre,0.0,0.000000,0.466586,0.560440,0.806122,0.917342,-0.013469,0.371659,0.730980,0.549733,0.000078,0.074074,0.346021,0.056429,0,1
9467,286,Tolima,0.0,0.000000,0.619684,0.576923,0.948980,0.823339,-0.005871,0.396903,0.782106,0.462032,0.000078,0.037037,0.173010,0.101348,0,0
9468,286,Valle del Cauca,4969.0,8.510974,0.877278,0.824176,0.714286,1.051864,0.072181,0.798473,0.997565,0.520856,0.000078,0.000000,0.000000,0.620719,0,1
9469,286,Vaupés,0.0,0.000000,-0.156258,-0.005495,0.051020,-0.108590,1.241754,0.224650,0.035301,-0.167914,0.000078,0.000000,0.000000,0.001404,1,0
9470,286,Vichada,0.0,0.000000,0.998785,0.620879,0.775510,1.043760,0.020376,0.594188,0.833841,0.479144,0.000078,0.000000,0.000000,0.005053,1,0


###Data exploration

In [ ]:
biweek = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/biweek_data_standardized.csv")
month = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/month_data_standardized.csv")
week = pd.read_csv("/content/drive/MyDrive/Fall '24/COS 397 ML IW/Final Data/11_12 Final/week_data_standardized.csv")

In [ ]:
week.head()

,week_number,Department,Weekly Quantity Seized (kg),Log Weekly Quantity Seized (kg),Percent Population in Urban Areas,HDI,GDI,Mean International Wealth Index,Infant Deaths per 1000 Live Births,Mean Years Schooling,Log Gross National Income Per Capita in 1000 USD,Life Expectancy,Population Density,Number of Corruption Cases,Percent of Total Corruption Cases for That Year,Number of Homicides that Year,Border_Department,Ocean_department
0,0,Amazonas,0.000,0.000000,-1.613379,-1.644489,-2.221961,-1.720798,0.643927,-0.531722,-1.745875,-1.340074,-0.324089,-0.389053,-0.260388,-0.662264,1,0
1,0,Antioquia,28.665,3.355677,0.462456,0.295543,0.280588,0.359676,-0.366227,0.058038,0.552396,0.164933,-0.324089,0.199369,0.644118,4.536211,0,1
2,0,Arauca,0.000,0.000000,0.505368,-0.203322,0.614262,0.073946,-0.599340,-0.502880,0.230755,0.531607,-0.324089,-0.683264,-0.712641,-0.416973,1,0
3,0,"Archipiélago de San Andrés, Providencia y Sant...",0.000,0.000000,-0.047116,0.905267,0.113752,0.440038,-0.356514,2.019261,0.637193,0.110205,-0.324089,-0.683264,-0.712641,-0.652114,0,0
4,0,Atlántico,0.606,0.000000,1.438689,0.461831,-0.497983,0.243598,0.061146,0.896431,0.426664,-0.316669,-0.324089,-0.094842,0.191865,0.249541,0,1


In [ ]:
print(len(biweek['Department'].unique()))

33


In [ ]:
# Number of departments
departments = month['Department'].unique()
num_departments = 33
#@CITATION: CODE ADAPTED FROM CHAT GPT
# Set up a large figure with 3 subplots for each department
fig, axes = plt.subplots(nrows=num_departments, ncols=3, figsize=(15, num_departments * 3), sharex=True)

# Loop through each department to create the 3 plots (monthly, biweekly, weekly)
for i, department in enumerate(departments):

    # Monthly data plot
    month_data = month[month['Department'] == department]
    axes[i, 0].plot(month_data['month_number'], month_data['Log Monthly Quantity Seized (kg)'], marker='o')
    axes[i, 0].set_title(f'{department} - Log Monthly Quantity Seized (kg)')
    axes[i, 0].set_ylabel('Log Quantity')
    axes[i, 0].grid()
    avg_log_quantity_month = month_data['Log Monthly Quantity Seized (kg)'].mean()
    print(f"{department}, month avg log quantity seized: {avg_log_quantity_month}")

    # Biweekly data plot
    biweek_data = biweek[biweek['Department'] == department]
    axes[i, 1].plot(biweek_data['biweek_number'], biweek_data['Log BiWeekly Quantity Seized (kg)'], marker='o')
    axes[i, 1].set_title(f'{department} - Log BiWeekly Quantity Seized (kg)')
    axes[i, 1].set_ylabel('Log Quantity')
    axes[i, 1].grid()
    avg_log_quantity_biweek = biweek_data['Log BiWeekly Quantity Seized (kg)'].mean()
    print(f"{department}, biweek avg log quantity seized: {avg_log_quantity_biweek}")

    # Weekly data plot
    week_data = week[week['Department'] == department]
    axes[i, 2].plot(week_data['week_number'], week_data['Log Weekly Quantity Seized (kg)'], marker='o')
    axes[i, 2].set_title(f'{department} - Log Weekly Quantity Seized (kg)')
    axes[i, 2].set_ylabel('Log Quantity')
    axes[i, 2].grid()
    avg_log_quantity_week = week_data['Log Weekly Quantity Seized (kg)'].mean()
    print(f"{department}, week avg log quantity seized: {avg_log_quantity_week}")

# Set x-axis label for the last row of plots
for ax in axes[-1, :]:
    ax.set_xlabel('Time Period')

plt.tight_layout()
plt.show()


Output hidden; open in https://colab.research.google.com to view.